# 02 — Analiza błędów (W3 §1.3)

Co mylą trzy ścieżki, dlaczego i czy to błąd modelu, czy niejednoznaczność danych.
Analiza i wnioski — autor; wczytywanie danych — `src/banking77_intent_ml/evaluation/error_analysis.py`.

**Zasady (W3.md §5):**
- **Test:** tylko agregaty z runów `split=test` — macierz 77×77, tabela per klasa, predykcje po `row_id`
  (bez tekstów). Zdań testowych nie czytamy, także po ocenie.
- **Walidacja:** przykłady błędów z modeli **W2** (trening na 8 504) — finalne modele widziały walidację.
- Etykiet nie zmieniamy; Ying & Thomas 2022 (~14% wskazań automatycznych) to kontekst, nie dowód błędu anotatora.
- Test ma 40 przykładów na klasę: 1 pomyłka = 2,5 p.p. recallu klasy — bez nadinterpretacji różnic per klasa.

In [ ]:
from banking77_intent_ml.config import load_settings
from banking77_intent_ml.evaluation.error_analysis import (
    add_label_names,
    plot_confusion,
    test_artifacts,
    val_predictions_w2,
)

settings = load_settings()
test = test_artifacts(settings)     # ścieżka -> TestArtifacts: confusion, per_class, predictions
val = val_predictions_w2(settings)  # ścieżka -> predykcje modeli W2 na walidacji (z tekstami)
{name: (artifacts.run_id[:8], len(artifacts.predictions)) for name, artifacts in test.items()}

## 1. Najczęstsze pary pomyłek (test)

- Które pary *prawdziwa → przewidziana* powtarzają się najczęściej w każdej ścieżce?
- Czy te same pary wracają we wszystkich trzech ścieżkach?
- Czy pomyłki są symetryczne (A → B i B → A)?

In [ ]:
plot_confusion(test["distilbert"].confusion, title="DistilBERT — test, wiersze znormalizowane");

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

card_pair_counts = {}
for name, artifacts in test.items():
    cm = artifacts.confusion.copy()
    np.testing.assert_array_equal(cm.index, cm.columns)
    for label in cm.index:
        cm.loc[label, label] = 0

    pairs = cm.rename_axis(index="prawdziwa", columns="przewidziana").stack()
    top_pairs = (
        pairs[pairs > 0]
        .nlargest(5, keep="all")
        .rename("liczba")
        .reset_index()
    )
    top_pairs["odwrotnie"] = [
        int(cm.loc[predicted, actual])
        for actual, predicted in zip(
            top_pairs["prawdziwa"], top_pairs["przewidziana"], strict=True
        )
    ]
    print(name)
    display(top_pairs)

    card_pair_counts[name] = int(
        cm.loc["card_arrival", "card_delivery_estimate"]
        + cm.loc["card_delivery_estimate", "card_arrival"]
    )

display(pd.Series(card_pair_counts, name="karta: oba kierunki").to_frame())

DistilBERT najczęściej myli oczekujący przelew z brakiem aktualizacji salda. Sześć przypadków, w przeciwnym kierunku trzy.
kNN najczęściej myli nieudane doładowanie z cofniętym. Sześć przypadków, w przeciwnym kierunku trzy.
TF-IDF ma dwie najczęstsze pomyłki. Nieudane doładowanie jako cofnięte oraz pytanie o powód weryfikacji tożsamości jako pytanie o sposób weryfikacji. Po pięć przypadków. W przeciwnych kierunkach po cztery.
Wszystkie modele mylą dostarczenie karty z przewidywanym terminem dostawy. Łącznie w obu kierunkach siedem pomyłek DistilBERT, osiem kNN i cztery TF-IDF. Pomyłki dotyczą bliskich intencji, ale nie zawsze są symetryczne.

## 2. Klasy z najniższym recallem (test)

- Które klasy mają najniższy recall w każdej ścieżce i czy to te same klasy?
- Ile to pomyłek w liczbach bezwzględnych (support 40 na klasę)?

In [ ]:
lowest_recall = {}
for name, artifacts in test.items():
    table = artifacts.per_class.copy()
    hits = pd.Series(
        np.diag(artifacts.confusion),
        index=artifacts.confusion.index,
    )
    table["błędy"] = (table["support"] - hits).astype(int)
    lowest_recall[name] = table.loc[
        table["recall"].eq(table["recall"].min()),
        ["recall", "support", "błędy"],
    ]

display(pd.concat(lowest_recall, names=["model", "klasa"]))

DistilBERT najsłabiej rozpoznaje odrzucone przelewy. Dziesięć błędów na 40 przykładów, recall 75 procent.
kNN najsłabiej rozpoznaje nierozpoznane płatności kartą i oczekujące przelewy. Po dziesięć błędów, recall 75 procent.
TF-IDF najsłabiej rozpoznaje niedziałającą kartę wirtualną. Jedenaście błędów, recall 72,5 procent.
Najsłabsze klasy są różne. Każda ma tylko 40 przykładów. Jedna pomyłka zmienia recall o 2,5 punktu procentowego.

## 3. Porównanie błędów między ścieżkami (test, po `row_id`)

- Ile przykładów myli każda ścieżka; które myli tylko TF-IDF+LR, a które wszystkie trzy?
- Czy błędy mają niższy `score` niż trafienia? (pamiętaj: przy k = 1 `score` kNN to zawsze 1,0)

In [ ]:
test_frames = {
    name: artifacts.predictions.set_index("row_id", verify_integrity=True).sort_index()
    for name, artifacts in test.items()
}
reference = test_frames["distilbert"]
for frame in test_frames.values():
    pd.testing.assert_index_equal(frame.index, reference.index)
    pd.testing.assert_series_equal(frame["y_true"], reference["y_true"])

wrong = pd.DataFrame({
    name: frame["y_true"].ne(frame["y_pred"])
    for name, frame in test_frames.items()
})
predicted = pd.DataFrame({
    name: frame["y_pred"] for name, frame in test_frames.items()
})

summary = {}
for name, frame in test_frames.items():
    mask = wrong[name]
    only_this = mask & ~wrong.drop(columns=name).any(axis=1)
    summary[name] = {
        "błędy": int(mask.sum()),
        "tylko ta ścieżka": int(only_this.sum()),
        "mediana score: błędy": frame.loc[mask, "score"].median(),
        "mediana score: trafienia": frame.loc[~mask, "score"].median(),
        "błędy ze score > 0.9": int((mask & frame["score"].gt(0.9)).sum()),
    }

display(pd.DataFrame.from_dict(summary, orient="index"))
all_wrong = wrong.all(axis=1)
same_prediction = predicted.nunique(axis=1).eq(1)
print("Wspólne błędy:", int(all_wrong.sum()))
print("Wspólne błędy z tą samą klasą:", int((all_wrong & same_prediction).sum()))

DistilBERT popełnia 215 błędów, kNN 245, a TF-IDF 283.
Wszystkie modele mylą się na 99 tych samych przykładach. W 39 przypadkach wybierają tę samą błędną klasę. Tylko TF-IDF myli się na 108 przykładach, tylko kNN na 89, a tylko DistilBERT na 55.
Błędne odpowiedzi mają zwykle niższy score. Mediana dla DistilBERT wynosi 0,676 przy błędach i 0,985 przy trafieniach. Dla TF-IDF odpowiednio 0,514 i 0,978.
Wysoki score nie gwarantuje poprawności. DistilBERT ma 57 błędów ze score powyżej 0,9, a TF-IDF 23. W kNN score zawsze wynosi 1,0, także przy błędach.

## 4. Przegląd błędnych predykcji z walidacji (modele W2)

- 10–20 błędów (np. wspólnych dla trzech ścieżek): czy tekst pasuje do obu intencji?
- Czy pomyłki skupiają się w rodzinach intencji (`card_*`, `top_up_*`, …)?

In [ ]:
errors = {
    name: add_label_names(frame, settings).query("y_true != y_pred") for name, frame in val.items()
}
{name: len(frame) for name, frame in errors.items()}

In [ ]:
val_frames = {
    name: add_label_names(frame, settings)
    .set_index("row_id", verify_integrity=True)
    .sort_index()
    for name, frame in val.items()
}
reference = val_frames["distilbert"]
for frame in val_frames.values():
    pd.testing.assert_index_equal(frame.index, reference.index)
    pd.testing.assert_series_equal(frame["y_true"], reference["y_true"])
    pd.testing.assert_series_equal(frame["text"], reference["text"])

val_wrong = pd.DataFrame({
    name: frame["y_true"].ne(frame["y_pred"])
    for name, frame in val_frames.items()
})
groups = [("wszystkie trzy", val_wrong.all(axis=1), 10)]
for name, count in [("tfidf_lr", 5), ("minilm_knn", 3), ("distilbert", 2)]:
    only_this = val_wrong[name] & ~val_wrong.drop(columns=name).any(axis=1)
    groups.append((f"tylko {name}", only_this, count))

review = reference[["text", "true_name"]].copy()
for name, frame in val_frames.items():
    review[name] = frame["pred_name"]

selected = pd.concat([
    review.loc[mask].head(count).assign(grupa=group)
    for group, mask, count in groups
])
assert len(selected) == 20
assert selected.index.is_unique

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(selected.reset_index())

Wybrano pierwsze przykłady według rosnącego row_id. Dziesięć błędów wspólnych dla trzech modeli. Pięć błędów tylko TF-IDF, trzy tylko kNN i dwa tylko DistilBERT. Dobór celowy, bez oceny częstości poszczególnych przyczyn błędów.
Przegląd obejmuje 20 przykładów z walidacji modeli W2. Dobór celowy, nie losowy. Poniżej wybrane obserwacje. Numery oznaczają row_id.
Przykład 32. Karta zagubiona podczas dostawy. TF-IDF wybiera zgubienie lub kradzież karty.
Przykład 245. Ponowne podłączenie wcześniej zgubionej karty. TF-IDF ponownie wybiera zgubienie lub kradzież. Możliwy zbyt duży wpływ słów o zagubieniu.
Przykład 919. Ogólne pytanie o akceptowane waluty. Brakuje informacji o rodzaju operacji. Wszystkie modele wybierają inną klasę niż etykieta.
Przykład 1131. Pytanie o niski stan salda, oznaczone jako automatyczne doładowanie. Doładowanie nie jest nazwane wprost. Wszystkie modele wybierają brak aktualizacji salda.
Przykład 1490. Pytanie o status nowej karty, oznaczone jako zgubienie lub kradzież. Wszystkie modele wybierają dostarczenie karty. Sam tekst nie wyjaśnia etykiety.
Przykład 1717. Reset PIN oznaczony jako zablokowany PIN. Wszystkie modele wybierają zmianę PIN. Granica intencji nie jest jednoznaczna.
Przykład 47. Oczekiwanie na kartę. kNN wybiera przewidywany termin dostawy. Obie intencje są bliskie znaczeniowo.
Przykład 387. Pytanie o kurs w aplikacji. kNN wybiera wymianę walut w aplikacji. Rozpoznaje temat, ale myli cel pytania.
Przykład 228. Ponowne podłączenie odnalezionej karty. DistilBERT wybiera zgubienie lub kradzież. Możliwe skupienie na wcześniejszym zdarzeniu.
Przykład 1008. Pytanie o dostawę karty do USA. DistilBERT wybiera obsługiwane kraje. Możliwy wpływ pobocznej informacji.
W przeglądzie powtarzają się podobne intencje, brak kontekstu i mylące fragmenty zdania. To możliwe wyjaśnienia błędów. Nie stanowią dowodu błędnej anotacji.

## 5. Odniesienie do Ying & Thomas 2022

- Które z Twoich par pomyłek pokrywają się z grupami wskazanymi jako niejednoznaczne lub błędnie etykietowane?
- Dlaczego nie poprawiamy etykiet?

Wspólne obszary trudności to dostarczenie karty, przelewy i doładowania. Autorzy wskazali około 14 procent potencjalnie problematycznych etykiet treningowych. To wskazania metod automatycznych, nie potwierdzony odsetek błędów anotacji.
Etykiety pozostają bez zmian. Zachowujemy zamrożony protokół i porównywalność wyników. Niezgodność predykcji z etykietą nie dowodzi, że etykieta jest błędna.

## 6. Wnioski (→ README, sekcja „Analiza błędów”)

Główna trudność to rozróżnianie bliskich intencji. Krótkie zdania często nie zawierają wystarczającego kontekstu.
Modele semantyczne poprawiają wynik ogólny względem TF-IDF. Nadal potrafią jednak skupić się na pobocznym fragmencie zdania.
DistilBERT popełnia najmniej błędów. Wysoki score nie gwarantuje poprawności, a score kNN przy k równym 1 nie informuje o pewności odpowiedzi.